In [1]:
# Cài đặt PySpark
%pip install pyspark

In [2]:
# Import các thư viện cần thiết
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import os

# Khởi tạo Spark Context
conf = SparkConf().setAppName("MovieRatingsAnalysis").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.appName("MovieRatingsAnalysis").getOrCreate()

print("Spark Context đã được khởi tạo thành công!")

Spark Context đã được khởi tạo thành công!


In [3]:
# Đọc dữ liệu từ các file
import os
from datetime import datetime

# Check if running in Google Colab
if 'COLAB_GPU' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    data_path = "/content/"
else:
    data_path = "data/" # For local environment

# Đọc file ratings_1.txt và ratings_2.txt
ratings_1_rdd = sc.textFile(data_path + "ratings_1.txt")
ratings_2_rdd = sc.textFile(data_path + "ratings_2.txt")

print(f"Số lượng rating từ file 1: {ratings_1_rdd.count()}")
print(f"Số lượng rating từ file 2: {ratings_2_rdd.count()}")

# Hiển thị một số dòng dữ liệu mẫu
print("\nDữ liệu ratings_1.txt (5 dòng đầu):")
for line in ratings_1_rdd.take(5):
    print(line)

print("\nDữ liệu ratings_2.txt (5 dòng đầu):")
for line in ratings_2_rdd.take(5):
    print(line)

Số lượng rating từ file 1: 84
Số lượng rating từ file 2: 100

Dữ liệu ratings_1.txt (5 dòng đầu):
7,1020,4.5,1577836800
23,1015,3.5,1577923200
45,1030,4.0,1578009600
12,1047,3.0,1578096000
38,1012,4.5,1578182400

Dữ liệu ratings_2.txt (5 dòng đầu):
12,1012,3.5,1577837800
34,1039,4.0,1577924200
27,1043,4.5,1578010600
8,1020,3.0,1578097000
19,1050,4.0,1578183400


In [4]:
# Xử lý dữ liệu ratings với timestamp để lấy năm
# Parse ratings: UserID, MovieID, Rating, Timestamp
def parse_rating_with_year(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])

    # Chuyển timestamp thành năm
    try:
        year = datetime.fromtimestamp(timestamp).year
    except:
        year = 2000  # Default year nếu không parse được

    return (year, rating)

# Parse cả 2 file ratings
ratings_1_parsed = ratings_1_rdd.map(parse_rating_with_year)
ratings_2_parsed = ratings_2_rdd.map(parse_rating_with_year)

print("Ratings 1 parsed with year (5 records):")
for rating in ratings_1_parsed.take(5):
    print(f"Year: {rating[0]}, Rating: {rating[1]}")

print("\nRatings 2 parsed with year (5 records):")
for rating in ratings_2_parsed.take(5):
    print(f"Year: {rating[0]}, Rating: {rating[1]}")

# Gộp 2 RDD ratings lại
all_ratings = ratings_1_parsed.union(ratings_2_parsed)
print(f"\nTổng số ratings từ cả 2 file: {all_ratings.count()}")

# Kiểm tra phạm vi năm
years_sample = all_ratings.map(lambda x: x[0]).distinct().collect()
years_sample.sort()
print(f"\nCác năm có trong dữ liệu: {years_sample}")
print(f"Năm sớm nhất: {min(years_sample)}")
print(f"Năm muộn nhất: {max(years_sample)}")

Ratings 1 parsed with year (5 records):
Year: 2020, Rating: 4.5
Year: 2020, Rating: 3.5
Year: 2020, Rating: 4.0
Year: 2020, Rating: 3.0
Year: 2020, Rating: 4.5

Ratings 2 parsed with year (5 records):
Year: 2020, Rating: 3.5
Year: 2020, Rating: 4.0
Year: 2020, Rating: 4.5
Year: 2020, Rating: 3.0
Year: 2020, Rating: 4.0

Tổng số ratings từ cả 2 file: 184

Các năm có trong dữ liệu: [2020]
Năm sớm nhất: 2020
Năm muộn nhất: 2020


In [5]:
# Tính tổng rating và số lượng rating cho mỗi năm
# (year, rating) -> (year, (rating, 1))
year_ratings_with_count = all_ratings.map(lambda x: (x[0], (x[1], 1)))

# Reduce theo key để tính tổng rating và tổng số lượng rating cho mỗi năm
# (year, (sum_ratings, total_count))
year_stats = year_ratings_with_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Tính điểm trung bình cho mỗi năm và hiển thị kết quả
def calculate_year_average(record):
    year, (sum_ratings, count) = record
    average_rating = sum_ratings / count
    return (year, (count, average_rating))  # (year, (total_ratings, average_rating))

year_results = year_stats.map(calculate_year_average)

# Sắp xếp theo năm tăng dần
sorted_year_results = year_results.sortByKey()

# Hiển thị tất cả các năm theo định dạng yêu cầu
all_years = sorted_year_results.collect()

for year, (total_ratings, avg_rating) in all_years:
    print(f"{year} - TotalRatings: {total_ratings}, AverageRating: {avg_rating:.2f}")

2020 - TotalRatings: 184, AverageRating: 3.75


In [6]:
# Dọn dẹp tài nguyên
sc.stop()
spark.stop()
print("Đã dừng Spark Context và Spark Session.")

Đã dừng Spark Context và Spark Session.
